## Problems 1

a) What Unicode character does `chr(0)` return?

In [8]:
print(chr(0) == '\0')

True


b) How does this character's string representation `(__repr__())` differ from its printed representation?

In [ ]:
print(chr(0))
print(repr(chr(0)))

 
'\x00'


c) What happens when this character occurs in text?

In [15]:
print(chr(0))
print("this is a test" + chr(0) + "string")

 
this is a test string


## Problems 2

a) What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than UTF-16 or UTF-32? It may be helpful to compare the output of these encodings for various input strings.

In [ ]:
test_string = "こんにちは" # test with other strings

utf8_encoded = test_string.encode("utf-8")
utf16_encoded = test_string.encode("utf-16")
utf32_encoded = test_string.encode("utf-32")

print("utf8:", utf8_encoded)
print("utf16:", utf16_encoded)
print("utf32:", utf32_encoded)

print("utf8 list:", list(utf8_encoded))
print("utf16 list:", list(utf16_encoded))
print("utf32 list:", list(utf32_encoded))

print("test string len:", len(test_string))
print("utf8 len:", len(utf8_encoded))
print("utf16 len:", len(utf16_encoded))
print("utf32 len:", len(utf32_encoded))

utf8: b'\xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf'
utf16: b'\xff\xfeS0\x930k0a0o0'
utf32: b'\xff\xfe\x00\x00S0\x00\x00\x930\x00\x00k0\x00\x00a0\x00\x00o0\x00\x00'
utf8 list: [227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175]
utf16 list: [255, 254, 83, 48, 147, 48, 107, 48, 97, 48, 111, 48]
utf32 list: [255, 254, 0, 0, 83, 48, 0, 0, 147, 48, 0, 0, 107, 48, 0, 0, 97, 48, 0, 0, 111, 48, 0, 0]
test string len: 5
utf8 len: 15
utf16 len: 12
utf32 len: 24


Using utf8 is shorter than other encodings for latin alphabet.

b) Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string into
a Unicode string. Why is this function incorrect? Provide an example of an input byte string
that yields incorrect results.

In [22]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

# decode_utf8_bytes_to_str_wrong("こ".encode("utf-8")) # will produce error

# The function is incorrect because for a char like こ it will produce an error
# because it is a multibyte encoding

decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))
bytestring = "kこ".encode("utf-8")
print(bytestring)
print(len(bytestring))
print(bytestring[0])
print(bytestring[1])
print(bytestring[2])
print(bytestring[3])
print(bytes([bytestring[0]]))
print(bytes(bytestring[1:]).decode("utf-8"))


b'k\xe3\x81\x93'
4
107
227
129
147
b'k'
こ


c) Give a two byte sequence that does not decode to any Unicode character(s).

In [30]:
print(bytes([227, 129]))
print(bytes([227, 129]).decode("utf-8"))

b'\xe3\x81'


UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 0-1: unexpected end of data

The way utf8 encodes/decodes: https://en.wikipedia.org/wiki/UTF-8#Description

In [3]:
x = [1,2,3]
y = [3,2,1]
print(zip(x,y))

### Train BPE Tests

In [56]:
!uv run pytest tests/test_train_bpe.py

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.1, pluggy-1.6.0
rootdir: /home/janahmed/Desktop/code/assignment1-basics
configfile: pyproject.toml
plugins: jaxtyping-0.3.2
collected 3 items                                                              

tests/test_train_bpe.py::test_train_bpe_speed PASSED
tests/test_train_bpe.py::test_train_bpe PASSED
tests/test_train_bpe.py::test_train_bpe_special_tokens PASSED

============================== 3 passed in 2.88s ===============================


In [6]:
from src.train_bpe import train_bpe

bpe = train_bpe("data/TinyStoriesV2-GPT4-train.txt", 10000, ["<|endoftext|>"])

print(max(bpe[0].values(), key=len))

with open('bpe_output.txt', 'w') as f:
    f.write('Vocabulary = ' + str(bpe[0]) + '\n')
    f.write('Merges = ' + str(bpe[1]) + '\n')

b' accomplishment'


In [1]:
import cProfile
from src.train_bpe import train_bpe

# bpe = train_bpe("data/TinyStoriesV2-GPT4-train.txt", 10000, ["<|endoftext|>"]) 
prof = cProfile.Profile()
bpe = prof.runcall(train_bpe, "data/TinyStoriesV2-GPT4-train.txt", 10000, ["<|endoftext|>"])
prof.print_stats(sort='cumulative') # Prints the report to your cell output

with open('bpe_output.txt', 'w') as f:
    f.write('Vocabulary = ' + str(bpe[0]) + '\n')
    f.write('Merges = ' + str(bpe[1]) + '\n')


         21414242 function calls (20667986 primitive calls) in 396.045 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
   102/98    0.001    0.000  355.404    3.627 connection.py:390(_recv)
  246/242    7.263    0.030  355.402    1.469 {built-in method posix.read}
       49    0.001    0.000  348.241    7.107 connection.py:246(recv)
       17    0.000    0.000  348.159   20.480 util.py:208(__call__)
        1    0.000    0.000  348.159  348.159 pool.py:680(_terminate_pool)
    51/49    0.000    0.000  348.143    7.105 connection.py:429(_recv_bytes)
        1    0.000    0.000  348.141  348.141 pool.py:671(_help_stuff_finish)
        1    0.000    0.000  348.141  348.141 {method 'acquire' of '_multiprocessing.SemLock' objects}
      3/1    0.000    0.000  348.141  348.141 threading.py:1018(_bootstrap)
      3/1    1.109    0.370  348.141  348.141 threading.py:1058(_bootstrap_inner)
      3/1    0.000    0.000  348.141  348

It took about 3.2GB of memory and without profiler it ran for about 20 seconds. The longest token in the vocabulary is `b' accomplishment'`.